In [1]:
pip install pandas numpy openpyxl xlrd scikit-learn xgboost imbalanced-learn shap matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

xls = pd.ExcelFile("CTG.xls")  
print(xls.sheet_names)

['Description', 'Data', 'Raw Data']


In [3]:
df = pd.read_excel("CTG.xls", sheet_name="Data")

print(df.shape)
print(df.columns.tolist())
print(df.head())
print(df.tail())


(2130, 46)
['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 22, 'Unnamed: 44', 23]
  Unnamed: 0 Unnamed: 1 Unnamed: 2 Unnamed: 3 Unnamed: 4 Unnamed: 5  \
0          b          e         AC         FM         UC         DL   
1        240        357          0          0          0          0   
2          5        632          4          0          4          2   
3        177        779          2          0          5          2   
4        411       1192          2          0          6          2   

  Unnamed: 6 Unnamed: 7 Unnamed: 8  Unnamed: 9  ... Unnamed: 36 Unnamed: 37  \
0         DS         DP         DR         NaN  ...           E 

In [4]:
df = pd.read_excel("CTG.xls", sheet_name="Data", header=1)

print(df.shape)
print(df.columns.tolist())
print(df.tail(6))

(2129, 46)
['b', 'e', 'AC', 'FM', 'UC', 'DL', 'DS', 'DP', 'DR', 'Unnamed: 9', 'LB', 'AC.1', 'FM.1', 'UC.1', 'DL.1', 'DS.1', 'DP.1', 'ASTV', 'MSTV', 'ALTV', 'MLTV', 'Width', 'Min', 'Max', 'Nmax', 'Nzeros', 'Mode', 'Mean', 'Median', 'Variance', 'Tendency', 'Unnamed: 31', 'A', 'B', 'C', 'D', 'E', 'AD', 'DE', 'LD', 'FS', 'SUSP', 'Unnamed: 42', 'CLASS', 'Unnamed: 44', 'NSP']
           b       e   AC     FM    UC    DL   DS   DP   DR  Unnamed: 9  ...  \
2123  1576.0  2596.0  1.0    0.0   7.0   0.0  0.0  0.0  0.0         NaN  ...   
2124  1576.0  3049.0  1.0    0.0   9.0   0.0  0.0  0.0  0.0         NaN  ...   
2125  2796.0  3415.0  1.0    1.0   5.0   0.0  0.0  0.0  0.0         NaN  ...   
2126     NaN     NaN  NaN    NaN   NaN   NaN  NaN  NaN  NaN         NaN  ...   
2127     NaN     NaN  NaN    NaN   NaN   0.0  0.0  0.0  0.0         NaN  ...   
2128     NaN     NaN  NaN  564.0  23.0  16.0  1.0  4.0  0.0         NaN  ...   

         E     AD     DE     LD    FS   SUSP  Unnamed: 42  CLASS  

In [5]:
# --- Step 1: Slice to real patient rows only (0–2125), drop the blank/footer rows below ---
df = df.iloc[0:2126].copy()

# --- Step 2: Drop blank spacer columns (not real data, just visual gaps in the original sheet) ---
spacer_cols = ['Unnamed: 9', 'Unnamed: 31', 'Unnamed: 42', 'Unnamed: 44']
df = df.drop(columns=spacer_cols)

# --- Step 3: Sanity checks ---
print(df.shape)
print(df.columns.tolist())
print(df.isnull().sum().sum())   # should be 0 now if slicing was correct
print(df['NSP'].value_counts())
print(df['CLASS'].value_counts())

(2126, 42)
['b', 'e', 'AC', 'FM', 'UC', 'DL', 'DS', 'DP', 'DR', 'LB', 'AC.1', 'FM.1', 'UC.1', 'DL.1', 'DS.1', 'DP.1', 'ASTV', 'MSTV', 'ALTV', 'MLTV', 'Width', 'Min', 'Max', 'Nmax', 'Nzeros', 'Mode', 'Mean', 'Median', 'Variance', 'Tendency', 'A', 'B', 'C', 'D', 'E', 'AD', 'DE', 'LD', 'FS', 'SUSP', 'CLASS', 'NSP']
0
NSP
1.0    1655
2.0     295
3.0     176
Name: count, dtype: int64
CLASS
2.0     579
1.0     384
6.0     332
7.0     252
10.0    197
8.0     107
4.0      81
5.0      72
9.0      69
3.0      53
Name: count, dtype: int64


In [6]:
df.duplicated().sum()

np.int64(10)

In [7]:
# --- Inspect the duplicates before deciding what to do ---
dupes = df[df.duplicated(keep=False)]
print(dupes.shape)
print(dupes.sort_values(by=list(df.columns)))

(19, 42)
           b       e   AC    FM   UC   DL   DS   DP   DR     LB  ...    C  \
326      0.0   809.0  0.0  15.0  0.0  0.0  0.0  0.0  0.0  144.0  ... -1.0   
333      0.0   809.0  0.0  15.0  0.0  0.0  0.0  0.0  0.0  144.0  ... -1.0   
229      0.0  1199.0  3.0   4.0  0.0  0.0  0.0  0.0  0.0  123.0  ... -1.0   
787      0.0  1199.0  3.0   4.0  0.0  0.0  0.0  0.0  0.0  123.0  ... -1.0   
67      10.0   697.0  5.0   0.0  3.0  0.0  0.0  0.0  0.0  140.0  ... -1.0   
68      10.0   697.0  5.0   0.0  3.0  0.0  0.0  0.0  0.0  140.0  ... -1.0   
790    176.0   928.0  2.0   3.0  0.0  0.0  0.0  0.0  0.0  123.0  ... -1.0   
791    176.0   928.0  2.0   3.0  0.0  0.0  0.0  0.0  0.0  123.0  ... -1.0   
323    179.0   666.0  0.0   0.0  0.0  0.0  0.0  0.0  0.0  135.0  ... -1.0   
324    179.0   666.0  0.0   0.0  0.0  0.0  0.0  0.0  0.0  135.0  ... -1.0   
233    276.0   596.0  0.0   0.0  0.0  0.0  0.0  0.0  0.0  123.0  ... -1.0   
234    276.0   596.0  0.0   0.0  0.0  0.0  0.0  0.0  0.0  123.0  ..

In [8]:
# --- Drop exact duplicate rows, keep first occurrence ---
before = df.shape[0]
df = df.drop_duplicates(keep='first').reset_index(drop=True)
after = df.shape[0]

print(f"Dropped {before - after} duplicate rows")
print(df.shape)   # expect (2117, 42)
print(df.duplicated().sum())   # expect 0

Dropped 10 duplicate rows
(2116, 42)
0


In [ ]:
# --- Save the cleaned dataset (2,116 rows) for backend/scripts/compute_feature_ranges.py ---
df.to_csv("fetal_clean.csv", index=False)

In [9]:
feature_cols = [
    'LB', 'AC.1', 'FM.1', 'UC.1', 'DL.1', 'DS.1', 'DP.1',
    'ASTV', 'MSTV', 'ALTV', 'MLTV',
    'Width', 'Min', 'Max', 'Nmax', 'Nzeros',
    'Mode', 'Mean', 'Median', 'Variance', 'Tendency'
]

X = df[feature_cols].copy()
y = df['NSP'].copy()

print(X.shape)      # expect (2116, 21)
print(y.shape)      # expect (2116,)
print(X.dtypes)
print(y.value_counts())
print(X.isnull().sum().sum())   # expect 0

(2116, 21)
(2116,)
LB          float64
AC.1        float64
FM.1        float64
UC.1        float64
DL.1        float64
DS.1        float64
DP.1        float64
ASTV        float64
MSTV        float64
ALTV        float64
MLTV        float64
Width       float64
Min         float64
Max         float64
Nmax        float64
Nzeros      float64
Mode        float64
Mean        float64
Median      float64
Variance    float64
Tendency    float64
dtype: object
NSP
1.0    1648
2.0     293
3.0     175
Name: count, dtype: int64
0


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(1692, 21) (424, 21)
NSP
1.0    0.778960
2.0    0.138298
3.0    0.082742
Name: proportion, dtype: float64
NSP
1.0    0.778302
2.0    0.139151
3.0    0.082547
Name: proportion, dtype: float64


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# --- Scaling (needed for LogisticRegression & SVM; RF/XGBoost don't need it, but no harm) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- XGBoost needs 0-indexed integer labels (NSP is 1/2/3) ---
y_train_xgb = y_train.astype(int) - 1
y_test_xgb = y_test.astype(int) - 1

results = {}

# 1. Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
results['Logistic Regression'] = lr.predict(X_test_scaled)

# 2. Random Forest
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
results['Random Forest'] = rf.predict(X_test)

# 3. SVM
svm = SVC(class_weight='balanced', random_state=42)
svm.fit(X_train_scaled, y_train)
results['SVM'] = svm.predict(X_test_scaled)

# 4. XGBoost (no direct class_weight param — use sample_weight instead)
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_xgb)
xgb = XGBClassifier(random_state=42, eval_metric='mlogloss')
xgb.fit(X_train, y_train_xgb, sample_weight=sample_weights)
xgb_preds = xgb.predict(X_test) + 1  # shift back to 1/2/3 to match others
results['XGBoost'] = xgb_preds

# --- Evaluate all 4 ---
for name, preds in results.items():
    print(f"\n===== {name} =====")
    print(classification_report(y_test, preds, target_names=['Normal', 'Suspect', 'Pathological']))
    print("Confusion Matrix (rows=actual, cols=predicted):")
    print(confusion_matrix(y_test, preds))
    print(f"Macro F1: {f1_score(y_test, preds, average='macro'):.4f}")


===== Logistic Regression =====
              precision    recall  f1-score   support

      Normal       0.98      0.89      0.93       330
     Suspect       0.56      0.81      0.67        59
Pathological       0.69      0.77      0.73        35

    accuracy                           0.87       424
   macro avg       0.75      0.83      0.78       424
weighted avg       0.90      0.87      0.88       424

Confusion Matrix (rows=actual, cols=predicted):
[[294  30   6]
 [  5  48   6]
 [  1   7  27]]
Macro F1: 0.7766

===== Random Forest =====
              precision    recall  f1-score   support

      Normal       0.97      0.97      0.97       330
     Suspect       0.83      0.85      0.84        59
Pathological       0.97      0.94      0.96        35

    accuracy                           0.95       424
   macro avg       0.93      0.92      0.92       424
weighted avg       0.95      0.95      0.95       424

Confusion Matrix (rows=actual, cols=predicted):
[[321   8   1]
 [  

In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Use full X, y (not just train) for CV, with sample weights for XGBoost applied per-fold via pipeline-less manual loop
# RF: class_weight handles imbalance natively per fold
rf_cv = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_scores = cross_val_score(rf_cv, X, y, cv=skf, scoring='f1_macro')

# XGBoost: needs 0-indexed labels; class_weight isn't native, so use sample_weight via a manual CV loop
y_xgb_full = y.astype(int) - 1
xgb_scores = []
for train_idx, test_idx in skf.split(X, y_xgb_full):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y_xgb_full.iloc[train_idx], y_xgb_full.iloc[test_idx]
    w = compute_sample_weight(class_weight='balanced', y=y_tr)
    model = XGBClassifier(random_state=42, eval_metric='mlogloss')
    model.fit(X_tr, y_tr, sample_weight=w)
    preds = model.predict(X_te)
    xgb_scores.append(f1_score(y_te, preds, average='macro'))
xgb_scores = np.array(xgb_scores)

print("Random Forest CV macro F1 per fold:", rf_scores)
print(f"Random Forest mean: {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})")
print()
print("XGBoost CV macro F1 per fold:", xgb_scores)
print(f"XGBoost mean: {xgb_scores.mean():.4f} (+/- {xgb_scores.std():.4f})")

Random Forest CV macro F1 per fold: [0.88304749 0.91286186 0.91895151 0.85908068 0.85256425]
Random Forest mean: 0.8853 (+/- 0.0270)

XGBoost CV macro F1 per fold: [0.88468003 0.93556544 0.93136818 0.89182553 0.89121121]
XGBoost mean: 0.9069 (+/- 0.0219)


In [13]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from xgboost import XGBClassifier
import numpy as np

# --- Recompute cleanly from X_train/y_train to avoid relying on stale variables ---
y_train_xgb = y_train.astype(int) - 1          # XGBoost needs 0-indexed labels (0,1,2)
y_test_xgb = y_test.astype(int) - 1

# Sample weights computed on the FULL X_train — RandomizedSearchCV's fit_params
# correctly slices this per CV fold internally (matches length of X_train)
sample_weights_train = compute_sample_weight(class_weight='balanced', y=y_train_xgb)

# --- Param search space ---
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

xgb_base = XGBClassifier(random_state=42, eval_metric='mlogloss')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1_macro',      # optimize for minority-class performance, not accuracy
    cv=skf,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train_xgb, sample_weight=sample_weights_train)

print("Best params:", search.best_params_)
print("Best CV macro F1:", search.best_score_)

# --- Evaluate the tuned model on the held-out test set ---
best_xgb = search.best_estimator_
test_preds = best_xgb.predict(X_test) + 1   # shift back to 1/2/3 labels

print("\n===== Tuned XGBoost — Test Set Performance =====")
print(classification_report(y_test, test_preds, target_names=['Normal', 'Suspect', 'Pathological']))
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_preds))
print(f"Macro F1: {f1_score(y_test, test_preds, average='macro'):.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 1.0, 'n_estimators': 500, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.1, 'gamma': 0.2, 'colsample_bytree': 0.8}
Best CV macro F1: 0.9216703341203626

===== Tuned XGBoost — Test Set Performance =====
              precision    recall  f1-score   support

      Normal       0.97      0.96      0.96       330
     Suspect       0.78      0.85      0.81        59
Pathological       0.97      0.97      0.97        35

    accuracy                           0.94       424
   macro avg       0.91      0.93      0.92       424
weighted avg       0.95      0.94      0.94       424

Confusion Matrix:
[[316  13   1]
 [  9  50   0]
 [  0   1  34]]
Macro F1: 0.9164


In [14]:
import shap

# --- TreeExplainer is the correct, exact (non-approximate) choice for XGBoost ---
explainer = shap.TreeExplainer(best_xgb)

# shap_values on the test set — for multiclass XGBoost this returns shape (n_samples, n_features, n_classes)
# in current SHAP versions (or a list of per-class arrays in older versions) — handle both safely
shap_values = explainer.shap_values(X_test)

print(type(shap_values))
if isinstance(shap_values, list):
    print(f"List of {len(shap_values)} arrays, each shape: {shap_values[0].shape}")
else:
    print(f"Array shape: {shap_values.shape}")

<class 'numpy.ndarray'>
Array shape: (424, 21, 3)


In [15]:
import matplotlib.pyplot as plt

class_names = ['Normal', 'Suspect', 'Pathological']

# --- Summary plots: save each class's plot to a file instead of just displaying ---
for i, cname in enumerate(class_names):
    shap.summary_plot(shap_values[:, :, i], X_test, feature_names=feature_cols, show=False)
    plt.title(f"SHAP Summary — {cname}")
    plt.tight_layout()
    plt.savefig(f"shap_summary_{cname.lower()}.png", dpi=150, bbox_inches='tight')
    plt.close()

print("Saved: shap_summary_normal.png, shap_summary_suspect.png, shap_summary_pathological.png")

Saved: shap_summary_normal.png, shap_summary_suspect.png, shap_summary_pathological.png


In [16]:
# --- Single-prediction waterfall plot: save to file ---
pathological_idx = np.where(y_test.values == 3)[0][0]
sample = X_test.iloc[[pathological_idx]]

pred_class = best_xgb.predict(sample)[0]
print(f"True label: Pathological | Predicted: {class_names[pred_class]}")

explanation = shap.Explanation(
    values=shap_values[pathological_idx, :, pred_class],
    base_values=explainer.expected_value[pred_class],
    data=sample.values[0],
    feature_names=feature_cols
)

shap.plots.waterfall(explanation, show=False)
plt.tight_layout()
plt.savefig("shap_waterfall_pathological_example.png", dpi=150, bbox_inches='tight')
plt.close()

print("Saved: shap_waterfall_pathological_example.png")

True label: Pathological | Predicted: Pathological
Saved: shap_waterfall_pathological_example.png


In [17]:
from sklearn.metrics import f1_score, classification_report

# --- Predictions on TRAIN set (using the already-fitted best_xgb) ---
train_preds = best_xgb.predict(X_train) + 1   # shift back to 1/2/3

print("===== Tuned XGBoost — TRAIN Set Performance =====")
print(classification_report(y_train, train_preds, target_names=['Normal', 'Suspect', 'Pathological']))
print(f"Train Macro F1: {f1_score(y_train, train_preds, average='macro'):.4f}")

print("\n===== Tuned XGBoost — TEST Set Performance (for comparison) =====")
test_preds = best_xgb.predict(X_test) + 1
print(classification_report(y_test, test_preds, target_names=['Normal', 'Suspect', 'Pathological']))
print(f"Test Macro F1: {f1_score(y_test, test_preds, average='macro'):.4f}")

print(f"\nGap (Train F1 - Test F1): {f1_score(y_train, train_preds, average='macro') - f1_score(y_test, test_preds, average='macro'):.4f}")

===== Tuned XGBoost — TRAIN Set Performance =====
              precision    recall  f1-score   support

      Normal       1.00      0.98      0.99      1318
     Suspect       0.92      1.00      0.96       234
Pathological       0.99      1.00      0.99       140

    accuracy                           0.99      1692
   macro avg       0.97      0.99      0.98      1692
weighted avg       0.99      0.99      0.99      1692

Train Macro F1: 0.9820

===== Tuned XGBoost — TEST Set Performance (for comparison) =====
              precision    recall  f1-score   support

      Normal       0.97      0.96      0.96       330
     Suspect       0.78      0.85      0.81        59
Pathological       0.97      0.97      0.97        35

    accuracy                           0.94       424
   macro avg       0.91      0.93      0.92       424
weighted avg       0.95      0.94      0.94       424

Test Macro F1: 0.9164

Gap (Train F1 - Test F1): 0.0655


In [18]:
# --- Step 1: Get real median profiles per class (grounded, not guessed) ---
class_profiles = df.groupby('NSP')[feature_cols].median()
print(class_profiles.T)  # transpose for readability

NSP              1.0         2.0         3.0
LB        132.000000  143.000000  132.000000
AC.1        0.002967    0.000000    0.000000
FM.1        0.000000    0.000000    0.000948
UC.1        0.004881    0.001497    0.003311
DL.1        0.000000    0.000000    0.002331
DS.1        0.000000    0.000000    0.000000
DP.1        0.000000    0.000000    0.001170
ASTV       41.000000   63.000000   65.000000
MSTV        1.300000    0.400000    1.700000
ALTV        0.000000   26.000000    0.000000
MLTV        7.950000    7.100000    3.200000
Width      71.000000   31.000000   91.000000
Min        90.000000  127.000000   66.000000
Max       163.000000  159.000000  157.000000
Nmax        4.000000    2.000000    4.000000
Nzeros      0.000000    0.000000    0.000000
Mode      138.000000  147.000000  122.000000
Mean      135.000000  146.000000  106.000000
Median    139.000000  148.000000  116.000000
Variance    9.000000    1.000000   37.000000
Tendency    0.000000    0.000000    0.000000


In [19]:
# --- Step 2: Test model on each class's median profile (the "textbook typical" case) ---
test_cases = class_profiles.T  # rows = feature, cols = class; transpose back for construction
test_cases = class_profiles[feature_cols]  # ensure correct column order matching training

for nsp_val, label in zip([1.0, 2.0, 3.0], class_names):
    sample = test_cases.loc[[nsp_val]]
    pred = best_xgb.predict(sample)[0] + 1
    proba = best_xgb.predict_proba(sample)[0]
    print(f"True class: {label} | Predicted: {class_names[int(pred)-1]} | Probabilities: {dict(zip(class_names, proba.round(3)))}")

True class: Normal | Predicted: Normal | Probabilities: {'Normal': np.float32(0.999), 'Suspect': np.float32(0.0), 'Pathological': np.float32(0.0)}
True class: Suspect | Predicted: Suspect | Probabilities: {'Normal': np.float32(0.018), 'Suspect': np.float32(0.978), 'Pathological': np.float32(0.004)}
True class: Pathological | Predicted: Pathological | Probabilities: {'Normal': np.float32(0.004), 'Suspect': np.float32(0.002), 'Pathological': np.float32(0.994)}


In [20]:
import pandas as pd

# --- Rebuild test predictions with class labels for inspection ---
test_preds = best_xgb.predict(X_test) + 1

results_df = X_test.copy()
results_df['True_NSP'] = y_test.values
results_df['Predicted_NSP'] = test_preds
results_df['True_Label'] = results_df['True_NSP'].map({1: 'Normal', 2: 'Suspect', 3: 'Pathological'})
results_df['Predicted_Label'] = results_df['Predicted_NSP'].map({1: 'Normal', 2: 'Suspect', 3: 'Pathological'})

# --- Isolate only the misclassified cases ---
errors = results_df[results_df['True_NSP'] != results_df['Predicted_NSP']]

print(f"Total misclassified: {len(errors)} out of {len(results_df)}")
print(errors[['True_Label', 'Predicted_Label'] + feature_cols])

Total misclassified: 24 out of 424
        True_Label Predicted_Label     LB      AC.1      FM.1      UC.1  \
531         Normal         Suspect  138.0  0.000834  0.000834  0.000000   
1465       Suspect          Normal  149.0  0.001705  0.000000  0.009378   
1313        Normal         Suspect  128.0  0.000000  0.012069  0.005172   
291         Normal         Suspect  142.0  0.000000  0.005128  0.000000   
437         Normal         Suspect  142.0  0.001136  0.003409  0.001136   
1804        Normal         Suspect  143.0  0.000000  0.000000  0.002907   
94          Normal         Suspect  148.0  0.000000  0.000000  0.000000   
436         Normal         Suspect  142.0  0.000834  0.002502  0.000834   
88         Suspect          Normal  139.0  0.001513  0.003026  0.000000   
722        Suspect          Normal  129.0  0.000000  0.007984  0.001996   
17         Suspect          Normal  131.0  0.001524  0.382622  0.003049   
1594       Suspect          Normal  140.0  0.000000  0.000000  0.

In [21]:
y_full_xgb = y.astype(int) - 1   # missing from the fixed version — re-add this

train_sizes_frac = np.linspace(0.1, 1.0, 8)
train_f1_scores = []
val_f1_scores = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for frac in train_sizes_frac:
    fold_train_f1 = []
    fold_val_f1 = []

    for train_idx, val_idx in skf.split(X, y_full_xgb):
        X_tr_full, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr_full, y_val = y_full_xgb.iloc[train_idx], y_full_xgb.iloc[val_idx]

        if frac >= 1.0:
            X_tr_sub, y_tr_sub = X_tr_full, y_tr_full
        else:
            n_samples = int(len(X_tr_full) * frac)
            X_tr_sub, _, y_tr_sub, _ = train_test_split(
                X_tr_full, y_tr_full,
                train_size=n_samples,
                stratify=y_tr_full,
                random_state=42
            )

        w = compute_sample_weight(class_weight='balanced', y=y_tr_sub)
        model = XGBClassifier(**search.best_params_, random_state=42, eval_metric='mlogloss')
        model.fit(X_tr_sub, y_tr_sub, sample_weight=w)

        train_preds = model.predict(X_tr_sub)
        val_preds = model.predict(X_val)

        fold_train_f1.append(f1_score(y_tr_sub, train_preds, average='macro'))
        fold_val_f1.append(f1_score(y_val, val_preds, average='macro'))

    train_f1_scores.append(np.mean(fold_train_f1))
    val_f1_scores.append(np.mean(fold_val_f1))

actual_sizes = (train_sizes_frac * len(X) * 0.8).astype(int)

plt.figure(figsize=(8,5))
plt.plot(actual_sizes, train_f1_scores, 'o-', label='Train F1 (macro)')
plt.plot(actual_sizes, val_f1_scores, 'o-', label='Validation F1 (macro)')
plt.xlabel('Training set size (approx.)')
plt.ylabel('Macro F1')
plt.title('Learning Curve — XGBoost (Fetal Risk Model)')
plt.legend()
plt.grid(True)
plt.savefig('learning_curve.png', dpi=150, bbox_inches='tight')
plt.close()

for size, tr, val in zip(actual_sizes, train_f1_scores, val_f1_scores):
    print(f"Train size: {size:4d} | Train F1: {tr:.4f} | Val F1: {val:.4f}")

print("\nSaved: learning_curve.png")

Train size:  169 | Train F1: 0.9818 | Val F1: 0.7794
Train size:  386 | Train F1: 0.9801 | Val F1: 0.8525
Train size:  604 | Train F1: 0.9795 | Val F1: 0.8741
Train size:  822 | Train F1: 0.9772 | Val F1: 0.8812
Train size: 1039 | Train F1: 0.9785 | Val F1: 0.8838
Train size: 1257 | Train F1: 0.9772 | Val F1: 0.8976
Train size: 1475 | Train F1: 0.9794 | Val F1: 0.8965
Train size: 1692 | Train F1: 0.9784 | Val F1: 0.8997

Saved: learning_curve.png


In [22]:
import joblib
import json
from datetime import datetime

# --- Save the trained model ---
joblib.dump(best_xgb, 'fetal_risk_xgboost_model.joblib')

# --- Save feature column order (critical: backend must match this exact order) ---
with open('fetal_risk_feature_order.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)

# --- Save label mapping (XGBoost predicts 0/1/2, needs mapping back to NSP labels) ---
label_mapping = {
    "model_output_to_nsp": {0: 1, 1: 2, 2: 3},
    "nsp_to_label": {1: "Normal", 2: "Suspect", 3: "Pathological"}
}
with open('fetal_risk_label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

# --- Save metadata for reference (not required by backend, but useful documentation) ---
metadata = {
    "model_type": "XGBoost",
    "task": "Fetal Risk Classification (Normal/Suspect/Pathological)",
    "trained_on": datetime.now().strftime("%Y-%m-%d"),
    "n_features": len(feature_cols),
    "best_params": search.best_params_,
    "test_macro_f1": 0.9164,
    "test_pathological_recall": 0.97,
    "test_suspect_recall": 0.85,
    "test_normal_recall": 0.96,
    "cv_macro_f1": 0.9217,
    "requires_scaling": False,
    "notes": "Uses class_weight='balanced' via sample_weight, no SMOTE. Feature order in fetal_risk_feature_order.json must be preserved exactly at inference time."
}
with open('fetal_risk_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved files:")
print("- fetal_risk_xgboost_model.joblib")
print("- fetal_risk_feature_order.json")
print("- fetal_risk_label_mapping.json")
print("- fetal_risk_model_metadata.json")

Saved files:
- fetal_risk_xgboost_model.joblib
- fetal_risk_feature_order.json
- fetal_risk_label_mapping.json
- fetal_risk_model_metadata.json


In [23]:
import os
print(os.listdir())

['.anaconda', '.antigravity-ide', '.conda', '.continuum', '.copilot', '.gemini', '.ipynb_checkpoints', '.ipython', '.jupyter', '.vscode', '.vscode-shared', 'anaconda3', 'anaconda_projects', 'AppData', 'Application Data', 'Contacts', 'Cookies', 'CTG.xls', 'Desktop', 'Documents', 'Downloads', 'Favorites', 'Fetal model.ipynb', 'fetal_risk_feature_order.json', 'fetal_risk_label_mapping.json', 'fetal_risk_model_metadata.json', 'fetal_risk_xgboost_model.joblib', 'IntelGraphicsProfiles', 'learning_curve.png', 'Links', 'Local Settings', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{997a41e4-5c92-11f1-819f-cc483a6f749f}.TM.blf', 'NTUSER.DAT{997a41e4-5c92-11f1-819f-cc483a6f749f}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{997a41e4-5c92-11f1-819f-cc483a6f749f}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'OneDrive', 'Pictures', 'PrintHood', 'Recent', 'Saved Games', 'Searches', 'SendTo', 'shap_summary_normal.png', '

In [24]:
loaded_model = joblib.load('fetal_risk_xgboost_model.joblib')
print(loaded_model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=0.2,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)


In [25]:
loaded_preds = loaded_model.predict(X_test) + 1
print(loaded_preds[:10])

[1 1 1 2 1 1 1 1 2 1]


In [26]:
original_preds = best_xgb.predict(X_test) + 1
print((loaded_preds == original_preds).all())

True


In [27]:
from sklearn.metrics import f1_score
print(f1_score(y_test, loaded_preds, average='macro'))

0.9164407325643594
